# EDA — Calificación de demandas de alimentos
**Iyotatsiri / SISGESC** · Notebook reproducible. No incluye resultados ficticios ni entrenamiento.


## 1. Carga y validación
Ejecutar desde la raíz del repositorio. La plantilla CSV comienza vacía: cargar únicamente registros seudonimizados y etiquetados por especialistas.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
BASE = Path.cwd()
if not (BASE / 'data').exists():
    BASE = BASE.parent
CSV = BASE / 'data/processed/casos_plantilla.csv'
df = pd.read_csv(CSV, dtype={'caso_id':'string','grupo_expediente':'string'})
print('Registros:',len(df),'Columnas:',len(df.columns))
print('Columnas:', ', '.join(df.columns))
if df.empty:
    print('SIN DATOS: completar CSV con expedientes anonimizados y validados antes de interpretar resultados.')


## 2. Integridad, faltantes y duplicados

In [ ]:
if not df.empty:
    display(df.isna().sum().sort_values(ascending=False).to_frame('faltantes'))
    print('IDs duplicados:', int(df['caso_id'].duplicated().sum()))
    print('Casos sin revisión experta:', int((df['revision_experto'].astype(str).str.upper()!='SI').sum()))
else:
    print('Análisis omitido: sin registros reales.')


## 3. Clases y balance de etiquetas

In [ ]:
if not df.empty and df['calificacion_validada'].notna().any():
    clases=df['calificacion_validada'].fillna('NO_DETERMINADA').value_counts()
    display(pd.DataFrame({'n':clases,'porcentaje':(100*clases/len(df)).round(2)}))
    ax=clases.plot.bar(title='Distribución de calificaciones validadas',ylabel='Expedientes',rot=25)
    plt.tight_layout(); plt.show()
else:
    print('Sin etiquetas validadas para analizar.')


## 4. Calidad de PDFs y necesidad de OCR

In [ ]:
if not df.empty:
    for col in ['paginas_demanda','paginas_resolucion']:
        print(col, pd.to_numeric(df[col],errors='coerce').describe().to_dict())
    for col in ['demanda_extraible','resolucion_extraible','requiere_ocr']:
        display(df[col].fillna('NO_DETERMINADO').value_counts().to_frame(col))
else:
    print('Sin documentos registrados.')


## 5. Variables jurídicas descriptivas
La ausencia documental no determina por sí sola una consecuencia procesal; cada regla requiere interpretación jurídica validada.

In [ ]:
juridicas=['petitorio_presente','partes_identificadas','firma_presente','anexos_identificados','vinculo_documentado','domicilio_demandado']
if not df.empty:
    tabla=pd.concat({c:df[c].fillna('NO_DETERMINADO').value_counts() for c in juridicas},axis=1).fillna(0).astype(int)
    display(tabla)
    tabla.T.plot(kind='bar',stacked=True,figsize=(11,5),title='Estados de variables jurídicas (descriptivo)')
    plt.tight_layout();plt.show()
else:
    print('Sin observaciones para variables jurídicas.')


## 6. Evolución temporal y control de fuga de información

In [ ]:
if not df.empty:
    df['fecha_ingreso_dt']=pd.to_datetime(df['fecha_ingreso'],errors='coerce')
    df['fecha_resolucion_dt']=pd.to_datetime(df['fecha_resolucion'],errors='coerce')
    print('Resoluciones anteriores al ingreso:',int(((df['fecha_resolucion_dt']<df['fecha_ingreso_dt']) & df['fecha_resolucion_dt'].notna()).sum()))
    serie=df.groupby(df['fecha_ingreso_dt'].dt.to_period('M')).size()
    if not serie.empty:
        serie.index=serie.index.astype(str)
        serie.plot(title='Expedientes por mes de ingreso',rot=45);plt.tight_layout();plt.show()
    print('ADVERTENCIA: no usar resolucion_pdf, motivo_observacion ni fundamentos tomados de la resolución como predictores.')
else:
    print('Sin fechas para explorar.')


## 7. Conclusiones y pasos siguientes
Al cargar registros reales, documentar tamaño muestral, balance de clases, faltantes, calidad OCR, duplicados, sesgos temporales y limitaciones. **No inferir desempeño predictivo desde el EDA.** Separar datos por expediente y tiempo antes de entrenar, mantener prueba final independiente y solicitar revisión jurídica de etiquetas y reglas.